# Data Modeling Deep Dive
## Kimball, Inmon, Data Vault 2.0, OBT & Modern Patterns

**What you'll learn:**
- Why data modeling matters (even in the lakehouse era)
- Normalization: 1NF through 3NF and when to denormalize
- Kimball dimensional modeling: star schema, facts, dimensions, conformed
- Inmon enterprise data warehouse: 3NF, subject-oriented
- Data Vault 2.0: hubs, links, satellites -- for agile warehousing
- One Big Table (OBT): when and why to flatten everything
- Activity Schema: event-centric modeling for modern analytics
- SCD Types 0-6: handling changing dimensions
- Modeling for analytics vs ML vs real-time
- Physical modeling: partitioning, clustering, sort keys
- Anti-patterns and how to fix them
- Interview questions with model design exercises

**Prerequisites:** 02_sql_foundation, 11_star_schema_modeling
**Estimated Time:** 8-10 hours

---

> "A good data model is worth 100 optimization hacks."
> The right model makes queries simple and fast.
> The wrong model makes everything painful forever.

---

---
# Section 1: Why Data Modeling Still Matters

## The Modeling Spectrum

```
RAW (no modeling)                                  HEAVY MODELING
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  Data Lake         Lakehouse           Data Warehouse        │
│  (schema-on-read)  (schema-on-write    (strict schemas,      │
│                     with flexibility)   normalization)        │
│                                                              │
│  "Dump everything" "Model the Gold     "Model everything     │
│                     layer well"          up front"            │
│                                                              │
│  Fast to ingest    Balanced             Slow to change        │
│  Hard to query     Best of both         Fast to query         │
│  No governance     Governed             Highly governed       │
└──────────────────────────────────────────────────────────────┘

MODERN BEST PRACTICE (2026):
  Bronze: No modeling (raw, as-is)
  Silver: Light modeling (clean, typed, deduplicated)
  Gold: FULL modeling (dimensional, optimized for consumers)
```

## Modeling Goals

| Goal | How Good Modeling Achieves It |
|------|------------------------------|
| **Query performance** | Pre-joined facts + dimensions = no complex JOINs at query time |
| **Understandability** | Business users understand "orders fact + customer dimension" |
| **Consistency** | Conformed dimensions = same customer definition everywhere |
| **Flexibility** | New questions answerable without new pipelines |
| **Governance** | Clear ownership, lineage, documentation per entity |
| **Cost** | Right grain + partitioning = scan less data |

---
# Section 2: Normalization & Denormalization

## Normal Forms (Quick Recap)

```
1NF: Atomic values (no arrays, no repeated groups)
  BAD:  | order_id | products          |
        | 1        | "Widget, Gadget"  |
  GOOD: | order_id | product           |
        | 1        | Widget            |
        | 1        | Gadget            |

2NF: 1NF + no partial dependencies (every non-key depends on FULL key)
  BAD:  | order_id | product_id | product_name | qty |
        (product_name depends only on product_id, not full PK)
  GOOD: Separate products table

3NF: 2NF + no transitive dependencies (non-key depends only on PK)
  BAD:  | employee_id | department_id | dept_name |
        (dept_name depends on department_id, not employee_id)
  GOOD: Separate departments table

BCNF: Every determinant is a candidate key (stricter 3NF)
```

## When to Normalize vs Denormalize

| Normalize (3NF) | Denormalize (Star Schema) |
|-----------------|--------------------------|
| OLTP (transactional systems) | OLAP (analytical systems) |
| Minimize write anomalies | Minimize query complexity |
| Source system / Bronze layer | Gold layer / Warehouse |
| Data integrity is priority | Query speed is priority |
| Many small updates | Large scans, aggregations |
| Storage is expensive | Compute is expensive |

## The Modern Reality

```
Source (3NF) → Bronze (raw) → Silver (light 3NF) → Gold (DENORMALIZED star)

In the Gold layer, INTENTIONALLY denormalize because:
  - Analysts don't want to write 7-table JOINs
  - Spark/Photon optimizes wide scans better than many JOINs
  - Pre-joined data = faster dashboards = happier stakeholders
  - Storage is cheap; analyst time is expensive
```

---
# Section 3: Kimball Dimensional Modeling

## The Star Schema

```
                    ┌─────────────┐
                    │ dim_date    │
                    │─────────────│
                    │ date_key PK │
                    │ date        │
                    │ year        │
                    │ quarter     │
                    │ month       │
                    │ day_of_week │
                    │ is_weekend  │
                    └──────┬──────┘
                           │
┌─────────────┐    ┌──────┴──────────────┐    ┌─────────────┐
│ dim_customer│    │   FACT_ORDERS       │    │ dim_product │
│─────────────│    │─────────────────────│    │─────────────│
│ cust_key PK │◄───│ order_id PK         │───►│ prod_key PK │
│ customer_id │    │ date_key FK         │    │ product_id  │
│ name        │    │ customer_key FK     │    │ name        │
│ segment     │    │ product_key FK      │    │ category    │
│ city, state │    │ channel_key FK      │    │ brand       │
│ ltv_band    │    │                     │    │ price_band  │
└─────────────┘    │ quantity     (measure)│   └─────────────┘
                    │ revenue      (measure)│
                    │ discount     (measure)│    ┌─────────────┐
                    │ shipping_cost(measure)│    │ dim_channel │
                    │                     │───►│─────────────│
                    └─────────────────────┘    │ channel_key │
                                               │ channel_name│
                                               │ device_type │
                                               └─────────────┘
```

## Kimball Key Concepts

| Concept | Description | Example |
|---------|-------------|---------|
| **Fact table** | Business process measurements (events) | fact_orders, fact_page_views |
| **Dimension table** | Context/descriptors (who/what/where/when) | dim_customer, dim_product, dim_date |
| **Grain** | Finest level of detail in fact table | One row per order line item |
| **Measure** | Numeric value to aggregate | revenue, quantity, discount |
| **Surrogate key** | Auto-generated dimension key (not business key) | dim_customer.customer_key (INT) |
| **Conformed dimension** | Shared across multiple fact tables | dim_date used by orders AND inventory |
| **Degenerate dimension** | Dimension stored in fact table (no separate table) | order_number in fact_order_lines |
| **Junk dimension** | Collection of low-cardinality flags | is_gift, is_express, is_international |

## Fact Table Types

| Type | Grain | Example | Measures |
|------|-------|---------|----------|
| **Transaction** | One row per event | fact_orders | revenue, qty (additive) |
| **Periodic snapshot** | One row per period | fact_monthly_balance | balance (semi-additive) |
| **Accumulating snapshot** | One row per lifecycle | fact_order_fulfillment | dates at each stage |
| **Factless** | Event occurrence (no measures) | fact_student_attendance | just the keys |

## Slowly Changing Dimensions (SCD)

| Type | Behavior | Implementation |
|------|----------|----------------|
| **Type 0** | Never changes | Original value preserved forever |
| **Type 1** | Overwrite | UPDATE in place (lose history) |
| **Type 2** | Add new row | effective_from, effective_to, is_current |
| **Type 3** | Add column | current_value + previous_value columns |
| **Type 4** | Separate history table | Current in main, history in archive |
| **Type 6** | Hybrid (1+2+3) | Current + history row + previous column |

In [ ]:
# Kimball modeling example with SQL
print("KIMBALL STAR SCHEMA -- SQL Example")
print("=" * 60)

print("""
-- FACT TABLE: One row per order line item (transaction grain)
CREATE TABLE gold.fact_order_lines (
    -- Keys
    order_line_id BIGINT GENERATED ALWAYS AS IDENTITY,
    order_id INT NOT NULL,
    date_key INT NOT NULL,          -- FK to dim_date
    customer_key INT NOT NULL,      -- FK to dim_customer (surrogate)
    product_key INT NOT NULL,       -- FK to dim_product (surrogate)
    
    -- Degenerate dimensions (no separate table needed)
    order_number STRING,
    payment_method STRING,
    
    -- Measures (what we aggregate)
    quantity INT,
    unit_price DECIMAL(10,2),
    line_revenue DECIMAL(12,2),     -- Additive: can SUM across any dimension
    discount_amount DECIMAL(10,2),  -- Additive
    shipping_cost DECIMAL(8,2),     -- Semi-additive: careful with double-counting
    
    -- Audit
    _loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
PARTITIONED BY (date_key);  -- Or use Liquid Clustering

-- DIMENSION TABLE: SCD Type 2 (track history)
CREATE TABLE gold.dim_customer (
    customer_key INT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
    customer_id INT NOT NULL,                        -- Natural/business key
    
    -- Attributes
    full_name STRING,
    email_domain STRING,      -- Not full email (PII minimization!)
    city STRING,
    state STRING,
    customer_segment STRING,
    lifetime_value_band STRING,  -- "High", "Medium", "Low"
    
    -- SCD Type 2 columns
    effective_from DATE NOT NULL,
    effective_to DATE DEFAULT '9999-12-31',
    is_current BOOLEAN DEFAULT TRUE,
    
    -- Audit
    _loaded_at TIMESTAMP
)
USING DELTA;

-- QUERY PATTERN (analyst-friendly):
-- "Revenue by customer segment and month"
SELECT
    d.year, d.month_name,
    c.customer_segment,
    SUM(f.line_revenue) AS revenue,
    COUNT(DISTINCT f.order_id) AS orders
FROM gold.fact_order_lines f
JOIN gold.dim_date d ON f.date_key = d.date_key
JOIN gold.dim_customer c ON f.customer_key = c.customer_key
WHERE c.is_current = TRUE
  AND d.year = 2024
GROUP BY d.year, d.month_name, c.customer_segment
ORDER BY d.month_name, revenue DESC;

-- NO complex subqueries! Clean, fast, understandable.
""")

---
# Section 4: Data Vault 2.0

## When Kimball Isn't Enough

Data Vault is for environments with:
- Many source systems feeding one warehouse
- Frequent source changes (agile/iterative development)
- Full audit trail requirement
- Need to load data FAST without complex business rules upfront

## Core Components

```
HUB (Business Key)         LINK (Relationship)        SATELLITE (Context)
┌─────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│ hub_customer    │       │ link_order      │       │ sat_customer    │
│─────────────────│       │─────────────────│       │─────────────────│
│ hub_cust_hk PK │◄──────│ link_order_hk PK│       │ hub_cust_hk FK │
│ customer_bk    │       │ hub_cust_hk FK  │──────►│ load_date PK   │
│ load_date      │       │ hub_prod_hk FK  │       │ record_source  │
│ record_source  │       │ load_date       │       │ name           │
└─────────────────┘       │ record_source   │       │ email          │
                          └─────────────────┘       │ segment        │
                                                    │ city           │
                                                    │ hash_diff      │
                                                    └─────────────────┘

HUB:  "This business entity EXISTS" (unique business keys)
LINK: "These entities are RELATED" (many-to-many relationships)
SAT:  "Here's the CONTEXT at a point in time" (attributes, full history)
```

## Data Vault vs Kimball

| Aspect | Data Vault | Kimball Star Schema |
|--------|-----------|-------------------|
| **Purpose** | Integration layer (raw vault) | Presentation layer (analytics) |
| **Loading** | Parallel, incremental (no dependencies) | Sequential (dims before facts) |
| **Schema changes** | Add new satellite (no existing changes) | May need schema migration |
| **Querying** | Complex (many JOINs) → needs Business Vault/mart on top | Simple (star JOIN) |
| **History** | Full history in satellites (automatic) | Only if SCD Type 2 |
| **Agility** | Very agile (add sources easily) | Requires upfront design |
| **Best for** | Enterprise DW with many sources | Analytics/BI direct access |

## Modern Usage

```
Typical modern architecture combining both:

Sources → Bronze (raw) → Silver (Data Vault: raw vault for integration)
                                         ↓
                              Gold (Kimball: star schemas for BI)
                              Gold (OBT: wide tables for DS/ML)
                              Gold (Metrics: pre-aggregated for dashboards)

Data Vault in Silver = flexible integration layer
Kimball in Gold = optimized consumption layer
```

---
# Section 5: Modern Data Modeling Patterns

## One Big Table (OBT)

```
Instead of:  fact_orders JOIN dim_customer JOIN dim_product JOIN dim_date
Build:       gold.orders_wide (all columns pre-joined, denormalized)

┌────────────────────────────────────────────────────────────────┐
│ gold.orders_wide                                                │
│────────────────────────────────────────────────────────────────│
│ order_id | order_date | customer_name | segment | city |        │
│ product_name | category | brand | quantity | revenue |           │
│ payment_method | channel | is_weekend | month_name | ...        │
│                                                                  │
│ 50+ columns, ZERO joins needed for most queries                  │
└────────────────────────────────────────────────────────────────┘

WHEN TO USE OBT:
  ✓ BI tools that struggle with JOINs (Looker, Metabase)
  ✓ Self-serve analytics (analysts don't know the schema)
  ✓ ML training datasets (features need to be in one table)
  ✓ Small-medium data (<100M rows) where storage is cheap

WHEN NOT TO USE:
  ✗ Extremely large data (duplication wastes storage)
  ✗ Many dimension updates (must rebuild entire table)
  ✗ Need different grains for different questions
```

## Activity Schema

```
A single table for ALL user activities (events-first modeling):

gold.activity_stream (
    activity_id STRING,
    activity_timestamp TIMESTAMP,
    customer_id STRING,           -- Who
    activity STRING,              -- What ("order_placed", "page_viewed", "support_ticket")
    
    -- Polymorphic attributes (different per activity type):
    anonymous_customer_id STRING,
    revenue_impact DECIMAL,
    feature_json STRING,          -- Flexible payload
    
    -- Context:
    activity_source STRING,       -- Where ("web", "mobile", "api")
    activity_medium STRING        -- How ("organic", "paid", "referral")
)
PARTITIONED BY (DATE(activity_timestamp))

BENEFITS:
  - All activities in ONE table (no hunting across 20 fact tables)
  - Easy to add new activity types (just new rows, no schema change)
  - Natural for funnel analysis, sessionization, attribution
  - Works well with BI tools that need a single source

DRAWBACKS:
  - Sparse columns (most activities don't have revenue_impact)
  - Harder to enforce types (feature_json is flexible but unstructured)
  - Aggregation requires CASE WHEN everywhere
```

## Modeling for Different Consumers

| Consumer | Best Model | Why |
|----------|-----------|-----|
| **BI Analyst** | Star schema or OBT | Simple queries, familiar patterns |
| **Data Scientist** | Wide table (features) | One row per entity, all features as columns |
| **ML Pipeline** | Feature table (point-in-time) | Timestamped features, no future data leakage |
| **Real-time app** | Denormalized, pre-aggregated | No JOINs at query time (latency) |
| **Finance** | Conformed dimensions, auditable | Consistent definitions, traceability |
| **Product team** | Activity schema / events | Flexible, funnel-friendly |

In [ ]:
# Data modeling decision framework
print("DATA MODELING DECISION FRAMEWORK")
print("=" * 60)

print("""
STEP 1: Identify the GRAIN (most important decision!)
  - What does ONE ROW represent?
  - "One row per order" vs "One row per order line item" vs "One row per day per product"
  - WRONG grain = wrong answers or impossible queries

STEP 2: Identify the FACTS (measures)
  - What numbers do we aggregate? (revenue, quantity, duration)
  - Additive? (SUM works across all dimensions)
  - Semi-additive? (SUM works across some, not all -- e.g., balance)
  - Non-additive? (ratios, percentages -- must recalculate)

STEP 3: Identify the DIMENSIONS (context)
  - Who? (customer, employee)
  - What? (product, service)
  - When? (date, time -- ALWAYS have a date dimension!)
  - Where? (location, store, region)
  - How? (channel, payment method)
  - Why? (promotion, campaign)

STEP 4: Handle CHANGES (SCD strategy)
  - Dimensions change! Customer moves city, product changes category.
  - Type 1 (overwrite): fast, simple, lose history
  - Type 2 (versioned): full history, complex, storage-heavy
  - Type 3 (prev column): limited history, simple queries

STEP 5: PHYSICAL optimization
  - Partitioning: by date (99% of queries filter by time)
  - Clustering/sort keys: by most-filtered dimensions
  - Compression: columnar (Parquet/Delta)
  - Pre-aggregation: materialized views for common rollups
""")

# Anti-patterns
print("\nDATA MODELING ANTI-PATTERNS:")
anti_patterns = [
    ("God table", "One massive table for everything", "Break into fact + dimensions"),
    ("No grain", "Mixed grains in same table", "Document and enforce ONE grain per table"),
    ("Premature aggregation", "Gold table only has monthly rollups", "Keep detail grain, add rollup as separate table"),
    ("Business keys as FK", "Join on email/name (mutable!)", "Use surrogate keys for joins, keep business key for lookup"),
    ("No date dimension", "Only raw timestamp in fact", "Create dim_date with fiscal year, holidays, weekday flags"),
    ("Circular references", "A references B references A", "Redesign with clear hierarchy"),
    ("NULL-heavy design", "50% of columns NULL most rows", "Consider polymorphic model or separate tables"),
]
print(f"  {'Anti-pattern':<25} {'Problem':<35} {'Fix'}")
print(f"  {'-'*90}")
for name, problem, fix in anti_patterns:
    print(f"  {name:<25} {problem:<35} {fix}")

In [ ]:
print("\nDATA MODELING INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "Design a data model for an e-commerce analytics platform.",
        "a": "Star schema. Facts: fact_orders (grain: one row per order line), fact_page_views (grain: one view). Dimensions: dim_customer (SCD2), dim_product, dim_date, dim_channel, dim_promotion. Grain is critical: order LINE (not order) to support product-level analysis. Partition fact by date. Conformed dim_customer used by both facts. Pre-aggregate gold.daily_revenue for dashboard speed."
    },
    {
        "q": "When would you choose Data Vault over Kimball?",
        "a": "Data Vault when: (1) many disparate sources being integrated, (2) source schemas change frequently, (3) full history/auditability is non-negotiable, (4) parallel loading is needed (no dependency between entities). Kimball when: (1) stable source systems, (2) clear business processes to model, (3) BI users query directly, (4) simpler architecture sufficient. Modern: often use both -- Data Vault in Silver, Kimball in Gold."
    },
    {
        "q": "How do you handle a dimension that changes frequently?",
        "a": "Depends on business need. If history doesn't matter: SCD Type 1 (overwrite). If history matters for analytics: SCD Type 2 (new row with effective dates). For high-frequency changes (e.g., user location updating every minute): consider a 'mini-dimension' -- extract frequently-changing attributes into a separate small dimension, update fact table FK. Or use Type 4 (current table + history table)."
    },
    {
        "q": "What's the difference between additive, semi-additive, and non-additive measures?",
        "a": "Additive: SUM works across ALL dimensions (revenue, quantity). Semi-additive: SUM works across SOME dimensions but not time (bank balance -- can't sum today's + yesterday's balance, but CAN sum across accounts). Non-additive: ratios and percentages (can't SUM conversion rates -- must recompute from components). Design tip: store components, derive ratios in the query."
    },
    {
        "q": "How do you model a many-to-many relationship in a star schema?",
        "a": "Use a bridge table (factless fact). Example: one order can have multiple promotions, one promotion applies to multiple orders. Create bridge_order_promotion(order_key, promotion_key, allocation_factor). The allocation_factor handles how to split credit (e.g., 50/50 for two promotions on one order). Join: fact → bridge → dim_promotion."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: Data Modeling Cheat Sheet

## When to Use Each Approach

| Approach | Layer | Best For | Complexity |
|----------|-------|----------|-----------|
| **3NF / Normalized** | Source / Silver | OLTP, write-heavy, data integrity | Medium |
| **Star Schema (Kimball)** | Gold | BI, analytics, dashboards | Medium |
| **Data Vault 2.0** | Silver | Enterprise integration, agile, audit | High |
| **OBT (One Big Table)** | Gold | Self-serve analytics, ML features | Low |
| **Activity Schema** | Gold | Event analytics, funnels | Low-Medium |
| **Pre-aggregated** | Gold | Dashboards, metrics layer | Low |

## The Golden Rules of Data Modeling

1. **Grain first**: Define what ONE ROW means before anything else
2. **Business alignment**: Model should match how users think about the domain
3. **Conformed dimensions**: Same customer definition across all facts
4. **Document everything**: Grain, SCD strategy, business rules
5. **Optimize for the consumer**: BI wants stars, ML wants wide, APIs want denormalized
6. **Physical = logical + optimization**: Partitioning, clustering, compression
7. **Evolve, don't rewrite**: Add dimensions/facts; avoid breaking existing

---
*Data Modeling Deep Dive for Data Engineers*